# Phase 3 — Data Cleaning

This notebook transforms the raw `DBtrainrides.csv` dataset into a
cleaned, analysis-ready DataFrame.

**Key transformations:**
1. Convert date/time columns from strings to datetime objects
2. Drop rows where arrival data is missing (origin stations)
3. Extract derived features: hour, weekday, month, season
4. Save the cleaned data to `data/cleaned/db_clean.parquet` for reuse

**Source data:** `data/DBtrainrides.csv` (2,061,357 rows)

In [1]:
import pandas as pd
import numpy as np
import os

# Load the raw data
df = pd.read_csv("../data/DBtrainrides.csv")
print("Raw data loaded.")
print("Shape:", df.shape)

Raw data loaded.
Shape: (2061357, 20)


In [2]:
# Columns that should be datetime
date_cols = ["arrival_plan", "departure_plan", "arrival_change", "departure_change"]

# Convert each one
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Verify the types changed
print("Data types after conversion:")
print(df[date_cols].dtypes)
print()
print("Sample values: ")
print(df[date_cols].head(3))

Data types after conversion:
arrival_plan        datetime64[us]
departure_plan      datetime64[us]
arrival_change      datetime64[us]
departure_change    datetime64[us]
dtype: object

Sample values: 
         arrival_plan      departure_plan      arrival_change  \
0 2024-07-08 00:00:00 2024-07-08 00:01:00 2024-07-08 00:03:00   
1                 NaT 2024-07-08 00:17:00                 NaT   
2 2024-07-08 00:03:00 2024-07-08 00:04:00 2024-07-08 00:03:00   

     departure_change  
0 2024-07-08 00:04:00  
1                 NaT  
2 2024-07-08 00:04:00  


In [3]:
# Count before
print(f"Rows before filtering: {len(df):,}")

# Keep only rows where we have a planned arrival time
df_clean = df.dropna(subset=["arrival_plan"]).copy()

# Count after
print(f"Rows after filtering: {len(df_clean):,}")
print(f"Dropped: {len(df) - len(df_clean):,} rows ({100*(len(df) - len(df_clean))/len(df):.1f}%)")

Rows before filtering: 2,061,357
Rows after filtering: 1,850,002
Dropped: 211,355 rows (10.3%)


In [4]:
# Hour of day (0–23)
df_clean["hour"] = df_clean["arrival_plan"].dt.hour

# Day of the week as a number (0 = Monday, 6 = Sunday)
df_clean["weekday_num"] = df_clean["arrival_plan"].dt.dayofweek

# Day of the week as a name ("Monday", "Tuesday", etc.)
df_clean["weekday_name"] = df_clean["arrival_plan"].dt.day_name()

# Month as a number (1–12)
df_clean["month"] = df_clean["arrival_plan"].dt.month

# Month as a name ("January", "February", etc.)
df_clean["month_name"] = df_clean["arrival_plan"].dt.month_name()

# Year
df_clean["year"] = df_clean["arrival_plan"].dt.year

# Season — using a custom function
def get_season(month):
 if month in [12, 1, 2]:
  return "Winter"
 elif month in [3, 4, 5]:
  return "Spring"
 elif month in [6, 7, 8]:
  return "Summer"
 else:
  return "Autumn"

df_clean["season"] = df_clean["month"].apply(get_season)

# Show a sample of the new columns
print("New time features:")
print(df_clean[["arrival_plan", "hour", "weekday_name", "month_name", "season"]].head(10))

New time features:
          arrival_plan  hour weekday_name month_name  season
0  2024-07-08 00:00:00     0       Monday       July  Summer
2  2024-07-08 00:03:00     0       Monday       July  Summer
3  2024-07-08 00:20:00     0       Monday       July  Summer
4  2024-07-08 00:20:00     0       Monday       July  Summer
5  2024-07-08 00:30:00     0       Monday       July  Summer
6  2024-07-08 00:58:00     0       Monday       July  Summer
7  2024-07-08 00:37:00     0       Monday       July  Summer
9  2024-07-08 00:27:00     0       Monday       July  Summer
10 2024-07-08 00:15:00     0       Monday       July  Summer
11 2024-07-08 00:39:00     0       Monday       July  Summer


In [5]:
def categorize_delay(minutes):
 if minutes <= 5:
  return "On time (≤5 min)"
 elif minutes <= 15:
  return "Slightly delayed (6–15 min)"
 elif minutes <= 30:
  return "Delayed (16–30 min)"
 else:
  return "Severely delayed (>30 min)"

df_clean["delay_category"] = df_clean["arrival_delay_m"].apply(categorize_delay)

# Check the distribution
print("Delay category distribution:")
print(df_clean["delay_category"].value_counts())
print()
print("As percentages:")
print(df_clean["delay_category"].value_counts(normalize=True).mul(100).round(2))

Delay category distribution:
delay_category
On time (≤5 min)               1739049
Slightly delayed (6–15 min)      89245
Delayed (16–30 min)              17481
Severely delayed (>30 min)        4227
Name: count, dtype: int64

As percentages:
delay_category
On time (≤5 min)               94.00
Slightly delayed (6–15 min)     4.82
Delayed (16–30 min)             0.94
Severely delayed (>30 min)      0.23
Name: proportion, dtype: float64


In [6]:
print("=== Cleaned dataset summary ===")
print(f"Shape: {df_clean.shape}")
print()
print("=== All columns ===")
print(df_clean.dtypes)
print()
print("=== Date range ===")
print(f"Earliest arrival: {df_clean['arrival_plan'].min()}")
print(f"Latest arrival:   {df_clean['arrival_plan'].max()}")

=== Cleaned dataset summary ===
Shape: (1850002, 28)

=== All columns ===
ID                                  str
line                                str
path                                str
eva_nr                            int64
category                          int64
station                             str
state                               str
city                                str
zip                               int64
long                            float64
lat                             float64
arrival_plan             datetime64[us]
departure_plan           datetime64[us]
arrival_change           datetime64[us]
departure_change         datetime64[us]
arrival_delay_m                   int64
departure_delay_m                 int64
info                                str
arrival_delay_check                 str
departure_delay_check               str
hour                              int32
weekday_num                       int32
weekday_name                        str
month 

In [7]:
# Create the cleaned data folder if it doesn't exist
os.makedirs("../data/cleaned", exist_ok=True)

# Save as Parquet
output_path = "../data/cleaned/db_clean.parquet"
df_clean.to_parquet(output_path, index=False)

# Verify it saved correctly by reading it back and checking shape
verification = pd.read_parquet(output_path)
print(f"Saved successfully to: {output_path}")
print(f"File shape on disk: {verification.shape}")
print(f"Datetime types preserved: {verification['arrival_plan'].dtype}")

Saved successfully to: ../data/cleaned/db_clean.parquet
File shape on disk: (1850002, 28)
Datetime types preserved: datetime64[us]
